# Day 08 — Automated Evaluation & Quality Gates

**Module 2 · The Metric Toolkit**

So far, we have run evaluations manually with `evaluate()`.

Today we turn an evaluation into an **automated test**.

The goal is simple:

```text
LLM Output
    ↓
Evaluation Metric
    ↓
Score
    ↓
Threshold
    ↓
PASS / FAIL


## 1. Setup

We will use OpenAI as the evaluation judge.

The judge is kept at `temperature=0` so that evaluation behavior is as consistent as possible.

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import OpenAIModel

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

print("Judge:", judge.get_model_name())

Judge: gpt-4.1-mini


## 2. Create an Evaluation

We will start with the same kind of test case used earlier in the course.

The important difference is that this time we want the evaluation to behave like a **test**.

In [2]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

test_case = LLMTestCase(
    input="What's your refund policy?",
    actual_output="Refunds are available within 30 days of purchase.",
)

metric = AnswerRelevancyMetric(
    model=judge,
    threshold=0.5,
)

print("Test case and metric created.")

Test case and metric created.


## 3. Turn the Evaluation into a Test

`assert_test()` connects a test case with one or more metrics.

If the metric passes its threshold, the test passes.

If the metric fails its threshold, the test fails.

This is the bridge between:

**LLM evaluation → automated testing**

In [3]:
from deepeval import assert_test

assert_test(
    test_case,
    metrics=[metric],
)

print("Evaluation passed.")

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Evaluation passed.


## 4. What Happens When the Answer Is Bad?

Let's intentionally give the model output that does not answer the question.

A good evaluation system should catch it automatically.


This cell is supposed to fail.

That's actually the lesson: the evaluation has become a quality gate.

In [4]:
bad_case = LLMTestCase(
    input="What's your refund policy?",
    actual_output="Our company was founded in 2018 and has offices around the world.",
)

assert_test(
    bad_case,
    metrics=[metric],
)

AssertionError: Metrics: Answer Relevancy (score: 0.0, threshold: 0.5, strict: False, error: None, reason: The score is 0.00 because the response provides information about the company's founding year and office locations, which do not address the refund policy question at all.) failed.

## 5. From Local Test to CI/CD

The same test can eventually run automatically when code changes:

```text
Developer changes prompt / model / RAG pipeline
                    ↓
              Run evaluation
                    ↓
              Metrics score
                    ↓
             Threshold check
               ↙         ↘
            PASS          FAIL
             ↓             ↓
          Merge          Block

# Day 08 — Key Takeaways

- `evaluate()` is useful for running evaluations and inspecting results.
- `assert_test()` turns an evaluation into a test.
- A metric threshold becomes a **quality gate**.
- A failing evaluation can stop an automated pipeline.
- Pytest is the testing infrastructure underneath this workflow, but we do not need to learn pytest deeply yet.
